## Topic 1: Why Chunking Exists

### Concept, Intuition, Why It Exists

- Two separate, unrelated problems both get solved by the same fix — splitting documents into smaller pieces.
- **Problem 1 — context window limits**: an LLM can only accept a fixed amount of text in one call. A 20-page policy PDF ingested as one giant Document (from the JSON loader) cannot be stuffed whole into a prompt alongside a user's question — it simply won't fit, and even where it technically fits, stuffing it in wastes most of the available space on irrelevant pages.
- **Problem 2 — retrieval precision**: even ignoring context limits entirely, retrieval itself gets worse with large units. If "one Document" = one entire 20-page PDF, a semantic search for "premature withdrawal penalty" returns the *whole* PDF as the best match — including 19 pages that have nothing to do with the question. The embedding for a giant block of mixed-topic text is a blurry average of everything in it, matching nothing precisely.
- Chunking is the fix for both at once: split each Document's `page_content` into smaller, focused pieces *before* embedding, so each piece (a) fits comfortably in a prompt, and (b) embeds to a vector that actually represents one coherent idea, not a blur of twenty.
- Important distinction from earlier loader work: text files, CSV/Excel, and JSON loaders already made *some* granularity decisions at the loader level — text files as whole files, CSV/Excel as one row, JSON as one page. Chunking is a **second**, independent splitting step that can still apply *within* any of those units when the unit itself is still too large or covers more than one topic (e.g. a single JSON page containing three unrelated policy clauses).

### Internal Working

- Conceptually, chunking sits immediately after ingestion and immediately before embedding: Document → chunker → smaller Documents → embedder → vector store.
- Each chunk still needs to carry the same `page_content` + `metadata` shape from the Document pattern — a chunker doesn't replace the Document abstraction, it just produces *more, smaller* Documents from each input Document. Metadata typically gets extended with a chunk index so a retrieved chunk can still be traced back to its position within the original page/row.
- The core tension every chunking strategy has to navigate: chunks too large drift back into Problem 2 (blurry embeddings, context waste); chunks too small lose coherence (a chunk that's half a sentence embeds to something meaningless, and retrieval returns fragments a human — or the LLM — can't actually use).

### How It's Implemented In This Project

- This project already has a working `chunk_text()` function that performs **sentence-aware** chunking — it doesn't split mid-sentence, which is already better than the simplest possible approach (fixed character count, ignoring word/sentence boundaries).
- Current gap, named explicitly here so it's not silently inherited going forward: `chunk_text()` currently has **zero overlap** between consecutive chunks. The next topic covers exactly why that's a real gap and how to fix it.

### Real-World Issues, Edge Cases, Debugging

- **Silent context overflow**: if chunking is skipped or misconfigured and a Document is too large, some embedding APIs truncate silently rather than erroring — the back half of a policy page simply never gets embedded, and nobody notices until a question about that exact content returns nothing relevant.
- **Cost compounds with chunk count**: more, smaller chunks mean more embedding calls and more vector store entries — chunking too aggressively isn't free either, it's a trade against Problem 2's precision, not a free win.
- **Debugging retrieval misses often starts here**: a confidently wrong or "no relevant info found" answer is very often a chunking problem, not a model problem — sample the actual chunks a query's top results returned and check whether they're coherent, complete thoughts before assuming the retriever or the LLM is at fault.

### Design Decisions & Trade-offs

- Chunking is treated as a separate stage from ingestion, not folded into it, mirroring the same separation-of-concerns reasoning used for loaders — ingestion's job is "normalize sources into Documents," chunking's job is "split Documents into retrieval-sized pieces." Keeping them separate means the chunking strategy can be changed later without touching a single loader.
- Sentence-aware chunking over naive fixed-character chunking is itself a deliberate trade: slightly more complex to implement, in exchange for chunks that are actually coherent units of meaning instead of arbitrary character-count cuts that can land mid-word or mid-sentence.

### Alternatives & When To Use Each

- Don't chunk at all, embed whole Documents — acceptable only when every Document is already small and single-topic (this is effectively what the text/CSV loaders already achieve at the loader level, which is *why* they deliberately chose small, single-topic units in the first place).
- Chunk every Document uniformly regardless of source — simplest to implement, but ignores that different source types already arrive at different natural granularities, so applies the same logic everywhere even where it isn't needed (the same point made earlier about not chunking the keyword reference files).
- Chunk selectively, only where a Document is still too large or multi-topic after ingestion — more deliberate, requires checking Document size/content before deciding, which is the realistic production approach.

### Common Mistakes & Production Failures

- Treating chunking as a one-time decision made early and never revisited, even as document types and sizes in the corpus change over time.
- Chunking uniformly across every source type without checking whether the loader-level granularity already solved the problem for that source.
- Not tracing a "no relevant chunk found" failure back to chunking at all — assuming it's a retrieval ranking or embedding model problem first, when the actual chunks were incoherent or too large from the start.

### Lead-Level Interview Questions

**Q: Why is chunking necessary even for an LLM with a very large context window?**
A: Context window size only solves Problem 1 (fitting text in). It does nothing for Problem 2 (retrieval precision) — even with unlimited context, embedding a 20-page document as one vector still produces a blurry, imprecise match for any specific question about one part of it. Chunking exists independently of context window size, for retrieval quality reasons alone.

**Q: A retrieval pipeline returns confidently wrong answers. Walk through how chunking specifically could be the root cause.**
A: If chunks are too large and multi-topic, their embeddings represent a blend of several ideas, so a query can match a chunk that's only tangentially related to the actual question, and the LLM ends up answering based on the wrong section of that chunk. If chunks are too small or cut mid-sentence, the retrieved text may be syntactically broken or missing the context needed to answer correctly even though the right general area was found. Sampling actual retrieved chunks for a known-bad query is the fastest way to confirm whether chunking, not the model, is the cause.

**Q: Why doesn't loader-level granularity (one Document per CSV row, whole file for text references) eliminate the need for chunking entirely?**
A: It eliminates the need for some sources, but not all. A single JSON page can still be long and cover multiple unrelated policy clauses — loader-level granularity solved the structural unit (one page = one Document), but didn't guarantee that unit is short or single-topic. Chunking is the safety net for sources where the natural loader-level unit still isn't small or focused enough.

### Hidden Concepts Worth Knowing

- **Token count vs. character count**: context window limits are usually expressed in tokens, not characters or words, and the relationship between them varies by language and content (code, non-English text, and rare words typically use more tokens per character). A chunking strategy that only counts characters can still silently overflow a token-based limit.
- **The "lost in the middle" effect**: even when content technically fits in context, LLMs are empirically less accurate at using information placed in the middle of a long context versus the beginning or end — another reason smaller, focused chunks tend to outperform one giant context dump even when the giant dump would technically fit.

### Revision Summary

> Chunking solves two separate problems at once: fitting text within context window limits, and keeping each retrievable unit focused enough that its embedding actually represents one coherent idea. It's a second, independent splitting step on top of whatever granularity ingestion already chose — necessary wherever a Document is still too large or multi-topic after loading. This project's `chunk_text()` is sentence-aware but currently has zero overlap, a gap covered next.